# Pawn counting

Project to explore/visualise chess pawn structures

In [38]:
import polars as pl
import numpy as np
import altair as alt

In [39]:
type Position = tuple[np.uint64, np.uint64]
"""
Pawn structure position --- a configuration of White and Black pawns on a board of size
BOARD_WIDTH * BOARD_DEPTH

Consists of two u64 bitmasks, the first for White pawns and the second for Black.

The least bit of each bitmask is the bottom-left square for White (a1), counting up in
file-major order (a1, a2, ..., a8, b1, ...) to the top-right square for White (h8).

If the board is smaller than 8x8, the board crops around bottom-left square for White
(so the bottom-left square remains a1), and the unused squares are empty.
"""

BOARD_WIDTH = 4
BOARD_DEPTH = 4

assert 0 <= BOARD_WIDTH <= 8
assert 0 <= BOARD_DEPTH <= 8

In [40]:
def position_ndarray(pos: Position) -> tuple[np.ndarray, np.ndarray]:
    def to_array(mask):
        arr = np.zeros((BOARD_WIDTH, BOARD_DEPTH), dtype=bool)
        for f in range(BOARD_WIDTH):
            for r in range(BOARD_DEPTH):
                arr[f, r] = (mask >> (f * 8 + r)) & 1
        return arr

    white, black = pos
    return to_array(white), to_array(black)

def init_position() -> Position:
    white = 0
    black = 0
    for f in range(BOARD_WIDTH):
        white |= 1 << (f * 8)
        black |= 1 << (f * 8 + BOARD_DEPTH - 1)
    return np.uint64(white), np.uint64(black)

def rand_position(max_pawns_per_side: int | None = None) -> Position:
    squares = [f * 8 + r for f in range(BOARD_WIDTH) for r in range(BOARD_DEPTH)]
    n = len(squares)
    cap = n if max_pawns_per_side is None else min(max_pawns_per_side, n)

    n_white = np.random.randint(cap + 1)
    n_black = np.random.randint(min(cap, n - n_white) + 1)

    chosen = np.random.choice(n, size=n_white + n_black, replace=False)
    white = 0
    black = 0
    for i in chosen[:n_white]:
        white |= 1 << squares[i]
    for i in chosen[n_white:]:
        black |= 1 << squares[i]
    return np.uint64(white), np.uint64(black)

def pawns_as_frame(pos: Position) -> pl.DataFrame:
    colour = pl.Enum(["White", "Black"])
    rows = []
    for name, mask in zip(["White", "Black"], pos):
        for f in range(BOARD_WIDTH):
            for r in range(BOARD_DEPTH):
                if (mask >> (f * 8 + r)) & 1:
                    rows.append({"rank": r + 1, "file": f + 1, "colour": name})
    return pl.DataFrame(rows, schema={"rank": pl.Int64, "file": pl.Int64, "colour": colour})

In [53]:
def position_chart(pos: Position) -> alt.LayerChart:
    _PAWN_COLOURS = {
        "domain": ["White", "Black"],
        "range": ["white", "black"],
    }
    _SQUARE_COLOURS = {
        "domain": ["light", "dark"],
        "range": ["#f0d9b5", "#b58863"],
    }
    _FILE_DOMAIN = list(range(1, BOARD_WIDTH + 1))
    _RANK_DOMAIN = list(range(1, BOARD_DEPTH + 1))

    def _board_as_frame() -> pl.DataFrame:
        rows = []
        for f in range(1, BOARD_WIDTH + 1):
            for r in range(1, BOARD_DEPTH + 1):
                square = "dark" if (f + r) % 2 == 0 else "light"
                rows.append({"rank": r, "file": f, "square": square})
        return pl.DataFrame(rows)

    board = (
        alt.Chart(_board_as_frame())
        .mark_rect()
        .encode(
            alt.X("file:O").scale(domain=_FILE_DOMAIN),
            alt.Y("rank:O").scale(domain=_RANK_DOMAIN),
            alt.Color("square:N").scale(**_SQUARE_COLOURS).legend(None),  # type: ignore
        )
    )

    pawns = (
        alt.Chart(pawns_as_frame(pos))
        .mark_circle(size=250, stroke="black", strokeWidth=0.5)
        .encode(
            alt.X("file:O").scale(domain=_FILE_DOMAIN),
            alt.Y("rank:O").scale(domain=_RANK_DOMAIN, reverse=True),
            alt.Color("colour:N")
            #
            .scale(**_PAWN_COLOURS),  # type: ignore
        )
    )

    return (
        alt.layer(board, pawns)
        .resolve_scale(color="independent")
        .properties(width=150, height=150)
    )  # type: ignore


position_chart(init_position())

alt.LayerChart(...)